# Golden Hour · Detection connected to Step 1
**Alex’s runnable notebook — updated for Abdurub’s actual cleaned and graph files.**

Place this notebook in the same shared folder as **`clean_transactions.csv`** and
**`graph_features.csv`**. Open this notebook, not the previous version, and run its
cells from top to bottom. The Python source is visible below; no separate `.py`
file, ZIP, raw data, private lookup, or secret key is needed.

**Default behaviour:** train and save a review copy to
`detection_outputs_step1/account_risk.csv`. It does **not** replace the shared
`account_risk.csv`, send anything to teammates, or freeze any account.

This version uses the full cleaned file, not `clean_transactions_sample.csv`.
It reads the existing graph CSV; it does not regenerate or overwrite Abdurub’s files.

**Evaluation boundary:** Step 1 selected all laundering-linked accounts and a sample
of other accounts, then included their neighbours. Results are retrospective
account-level out-of-fold development results on that label-enriched slice,
not proof of real-time or real-world detection. Full details remain in the report.

## 1 · Settings
The shared path below is taken from the uploaded Step 1 notebook; outside that
server, the current folder is used. Leave the settings unchanged for the first run.
One CPU thread and sequential folds limit contention. Do not run the old and new
detection notebooks simultaneously. The memory check is a precaution, not a guarantee
about LinuxONE memory use or runtime.

In [ ]:
from pathlib import Path
import os
import sys
if sys.version_info < (3, 9):
    raise RuntimeError("Use the team's Python 3.9+ kernel; do not replace the shared environment.")
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

server_shared = Path("/home/ibm-user/shared")
SHARED_DIR = server_shared if server_shared.is_dir() else Path.cwd()
INPUT_PATH = SHARED_DIR / "clean_transactions.csv"
GRAPH_PATH = SHARED_DIR / "graph_features.csv"
OUTPUT_DIR = SHARED_DIR / "detection_outputs_step1"

MODEL_BACKEND = "auto"  # XGBoost preferred; explicit scikit-learn fallback if unavailable.
N_SPLITS = 3
N_ESTIMATORS = 200
N_THREADS = 1
RANDOM_SEED = 42
MAX_TRANSACTIONS = 1_000_000  # Full supplied file has 832,742 rows; no silent truncation.
MIN_AVAILABLE_RAM_GIB = 1.5
TOP_K = 100
ALERT_FRACTION = 0.005
RUN_BASELINE_COMPARISON = True
MAKE_PLOTS = True

PUBLISH_TO_SHARED = True  # Publishes /shared/account_risk.csv when the final publishing cell runs.
print("Reading from:", SHARED_DIR)
print("Review-copy output:", OUTPUT_DIR / "account_risk.csv")
print("Replace shared account_risk.csv:", PUBLISH_TO_SHARED)

## 2 · Program functions
Run these cells in order. They define the actual program and do not train anything
yet. Dependencies are `numpy`, `pandas`, `scikit-learn`, `threadpoolctl`, and preferably
`xgboost`. `matplotlib` is optional. NetworkX is **not** needed here because Step 1
already calculated the graph. No packages are installed by this notebook.

### 2A · Input validation and memory-efficient loading

In [ ]:
"""Golden Hour: Alex's offline account-detection handoff.

Basis: Golden_Hour_Team_Plan.pdf, pp. 1-3. This implements that document's
account-level, out-of-fold CSV workflow, NOT the original spec's per-bank,
past-only transaction scorer. See the companion notebook for the distinction.

No raw data, HMAC key, fabricated performance, network calls, or accelerator use.
"""
from __future__ import annotations

# Apply before numerical libraries import. Runtime threadpool limits below also
# work when a notebook kernel has already imported NumPy or scikit-learn.
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                    "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = "1"

import argparse
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import platform
import re
import tempfile
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from threadpoolctl import threadpool_limits

REQUIRED_COLUMNS = [
    "tx_id", "timestamp", "from_acct", "to_acct", "from_bank", "to_bank",
    "amount_paid", "currency", "payment_format", "is_laundering",
]
HANDOFF_COLUMNS = ["acct", "risk_score", "is_laundering", "total_in", "total_out"]
AMOUNT_WARNING = (
    "total_in and total_out preserve the team-plan CSV contract: they sum "
    "amount_paid in original units, with NO FX conversion. Mixed-currency sums "
    "are NOT comparable money values. total_in is a payer-quoted incoming "
    "amount proxy, NOT confirmed Amount Received. Use the by-currency audit "
    "and agree currency handling before optimising money blocked."
)
EVALUATION_WARNING = (
    "Offline retrospective account-level OOF development evaluation. Each "
    "account row is excluded from the model that predicts it, but connected "
    "accounts can share transactions or schemes across folds. Features and "
    "labels describe the supplied complete snapshot. This is NOT time-held-out, "
    "scheme-held-out, bank-local, production, or early-intervention validation. "
    "Do not feed these snapshot scores into a past-only replay as though they "
    "were available at the earlier decision time."
)

@dataclass(frozen=True)
class ModelConfig:
    """Implementation defaults, not settings claimed by either source PDF."""
    backend: str = "auto"       # auto: XGBoost, with an explicitly logged fallback
    n_splits: int = 3             # fast first sprint run; choose before looking at scores
    n_estimators: int = 200
    max_depth: int = 4
    learning_rate: float = 0.08
    n_threads: int = 1
    random_state: int = 42
    top_k: int = 100
    alert_fraction: float = 0.005 # additional account budget; not a tuned threshold
    reference_threshold: float = 0.5

@dataclass
class AccountData:
    X: pd.DataFrame
    y: pd.Series
    accounts: pd.DataFrame
    amounts_by_currency: pd.DataFrame
    metadata: dict[str, Any]

@dataclass
class CVResult:
    account_risk: pd.DataFrame
    oof_audit: pd.DataFrame
    fold_metrics: pd.DataFrame
    budget_metrics: pd.DataFrame
    feature_importance: pd.DataFrame
    metrics: dict[str, Any]
    backend: str
    config: ModelConfig


def environment_report() -> dict[str, str]:
    result = {"python": platform.python_version(), "architecture": platform.machine()}
    for name in ("numpy", "pandas", "scikit-learn", "xgboost", "threadpoolctl"):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "not installed"
    result["accelerator"] = "not used or benchmarked by this pipeline"
    return result


def _file_signature(path: Path) -> tuple[int, int]:
    stat = path.stat()
    return stat.st_size, stat.st_mtime_ns


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _csv_header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        header = next(csv.reader(stream), [])
    if not header:
        raise ValueError(f"{path.name} has no CSV header.")
    if len(header) != len(set(header)):
        raise ValueError("Duplicate CSV column names. Ask the file owner to repair the header.")
    return header


def validate_transactions(frame: pd.DataFrame) -> pd.DataFrame:
    """Validate, never silently drop bad rows or rewrite account codes."""
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(
            f"Missing agreed clean columns: {missing}. Read Abdurub's clean_transactions.csv, "
            "not the raw IBM CSV. Do not rename the shared handoff on your own."
        )
    if frame.empty:
        raise ValueError("The clean transaction file is empty.")
    df = frame[REQUIRED_COLUMNS].copy()
    string_columns = ["tx_id", "from_acct", "to_acct", "from_bank", "to_bank",
                      "currency", "payment_format"]
    for name in string_columns:
        series = df[name].astype("string")
        if series.isna().any() or series.str.strip().eq("").any():
            raise ValueError(f"Missing/blank values in {name}; ask Abdurub to fix them.")
        if not series.eq(series.str.strip()).all():
            raise ValueError(f"Surrounding whitespace in {name}; fix upstream, not by changing IDs here.")
        df[name] = series
    if not df["tx_id"].str.fullmatch(r"\d+").all():
        raise ValueError("tx_id must contain non-negative integer row numbers.")
    # Treat 001 and 1 as the same row number when testing uniqueness.
    numeric_ids = pd.to_numeric(df["tx_id"], errors="raise")
    if numeric_ids.duplicated().any():
        raise ValueError("Repeated tx_id values. Do not double-count duplicated transactions.")
    amount = pd.to_numeric(df["amount_paid"], errors="coerce").astype("float64")
    if not np.isfinite(amount.to_numpy()).all() or amount.lt(0).any():
        raise ValueError("amount_paid must be finite and non-negative; no rows were dropped.")
    df["amount_paid"] = amount
    labels = pd.to_numeric(df["is_laundering"], errors="coerce")
    if not labels.isin([0, 1]).all():
        raise ValueError("is_laundering must be 0 or 1 without missing values.")
    df["is_laundering"] = labels.astype("int8")
    # Team plan gives a timestamp but no source timezone. The original spec's
    # contract is UTC. A naive input is therefore interpreted as UTC here;
    # confirm that convention with the data owner, rather than inferring a zone.
    parsed = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    if parsed.isna().any():
        raise ValueError("Unparseable/missing timestamps. Agree one timestamp format with Abdurub.")
    df["timestamp"] = parsed
    ownership = pd.concat([
        df[["from_acct", "from_bank"]].rename(columns={"from_acct": "acct", "from_bank": "bank"}),
        df[["to_acct", "to_bank"]].rename(columns={"to_acct": "acct", "to_bank": "bank"}),
    ], ignore_index=True).drop_duplicates()
    if ownership.groupby("acct", observed=True)["bank"].nunique().gt(1).any():
        raise ValueError(
            "At least one account code maps to more than one bank group. "
            "Ask Abdurub to check bank:account tokenisation and collisions. "
            "Do not repair IDs in Alex's notebook: Abdel must use identical codes."
        )
    # Reuse categorical dictionaries: important for the 832,742-row Step 1 file.
    account_categories = pd.Index(sorted(set(df["from_acct"]) | set(df["to_acct"])))
    bank_categories = pd.Index(sorted(set(df["from_bank"]) | set(df["to_bank"])))
    for col in ("from_acct", "to_acct"):
        df[col] = pd.Categorical(df[col], categories=account_categories)
    for col in ("from_bank", "to_bank"):
        df[col] = pd.Categorical(df[col], categories=bank_categories)
    for col in ("currency", "payment_format"):
        df[col] = df[col].astype("category")
    df["tx_id"] = numeric_ids.astype("int64")
    return df


def load_transactions(path: str | Path, max_transactions: int = 1_000_000) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Bound RAM without silently sampling a different dataset."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Not found: {path.resolve()}. Put Abdurub's clean_transactions.csv beside "
            "the notebook or change INPUT_PATH. No fake replacement was created."
        )
    if max_transactions < 1:
        raise ValueError("max_transactions must be positive.")
    signature = _file_signature(path)
    header = _csv_header(path)
    missing = sorted(set(REQUIRED_COLUMNS) - set(header))
    if missing:
        raise ValueError(f"Wrong input schema; missing {missing}. Expected the team's cleaned CSV.")
    # Read identifiers as strings: preserve leading zeroes and never learn from them.
    string_dtypes = {c: "string" for c in REQUIRED_COLUMNS if c != "amount_paid"}
    raw = pd.read_csv(
        path, usecols=REQUIRED_COLUMNS, dtype=string_dtypes,
        nrows=max_transactions + 1, keep_default_na=False, encoding="utf-8-sig",
    )
    if len(raw) > max_transactions:
        raise MemoryError(
            f"Input exceeds the explicit {max_transactions:,}-row safety limit. "
            "Nothing was silently truncated. Agree a suitable clean slice with "
            "Abdurub or raise MAX_TRANSACTIONS after checking shared RAM."
        )
    df = validate_transactions(raw)
    del raw
    fingerprint = _sha256(path)
    if _file_signature(path) != signature:
        raise RuntimeError("Input changed while it was being read. Ask the team to finish writing, then retry.")
    metadata = {
        "input_file": path.name, "input_resolved_path": str(path.resolve()),
        "input_sha256": fingerprint, "input_size_bytes": signature[0],
        "input_mtime_ns": signature[1], "ignored_extra_columns": sorted(set(header) - set(REQUIRED_COLUMNS)),
        "max_transactions_guard": max_transactions,
    }
    return df, metadata

### 2B · Account features and currency-separated amounts

In [ ]:
def build_account_features(df: pd.DataFrame, source_metadata: Optional[dict[str, Any]] = None) -> AccountData:
    """Deterministic snapshot aggregates; the feature builder never reads labels.

    y is created in a separate final block. No target encoding, counterparty
    labels, label-based graph weights, learned scaling, or ID-as-feature.
    Currency vocabularies are observed on the input snapshot without labels;
    this is a transductive/offline feature schema, not a future-stream schema.
    """
    # Validate at the public boundary; passing an already validated frame is safe.
    df = validate_transactions(df)
    accts = pd.Index(sorted(set(df["from_acct"]) | set(df["to_acct"])), name="acct")
    gb_out = df.groupby("from_acct", sort=False, observed=True)
    gb_in = df.groupby("to_acct", sort=False, observed=True)
    accounts = pd.DataFrame(index=accts)
    accounts["total_in"] = gb_in["amount_paid"].sum().reindex(accts, fill_value=0)
    accounts["total_out"] = gb_out["amount_paid"].sum().reindex(accts, fill_value=0)
    if not np.isfinite(accounts.to_numpy()).all():
        raise ValueError("Account amount sums overflowed. Check the input values.")

    # Incident activity counts a self-transfer once, while the separate flow
    # totals/counts correctly record one incoming AND one outgoing endpoint.
    obs_cols = ["tx_id", "timestamp", "amount_paid", "currency", "payment_format"]
    outgoing = df[obs_cols + ["from_acct", "to_acct"]].rename(
        columns={"from_acct": "acct", "to_acct": "counterparty"})
    incoming = df[obs_cols + ["to_acct", "from_acct"]].rename(
        columns={"to_acct": "acct", "from_acct": "counterparty"})
    cross_bank = df["from_bank"].ne(df["to_bank"]).astype("int8").to_numpy()
    outgoing["cross_bank"] = cross_bank
    incoming["cross_bank"] = cross_bank
    activity = pd.concat([outgoing, incoming], ignore_index=True).drop_duplicates(["acct", "tx_id"])
    del outgoing, incoming
    activity["cash"] = activity["payment_format"].str.casefold().eq("cash").astype("int8")
    activity["self_transfer"] = activity["acct"].eq(activity["counterparty"]).astype("int8")
    activity["date"] = activity["timestamp"].dt.floor("D")
    activity["hour_bucket"] = activity["timestamp"].dt.floor("h")
    agg = activity.groupby("acct", sort=False, observed=True)
    X = pd.DataFrame(index=accts)
    X["n_outgoing"] = gb_out.size().reindex(accts, fill_value=0)
    X["n_incoming"] = gb_in.size().reindex(accts, fill_value=0)
    X["n_transactions"] = agg.size().reindex(accts, fill_value=0)
    X["n_sent_to"] = gb_out["to_acct"].nunique().reindex(accts, fill_value=0)
    X["n_received_from"] = gb_in["from_acct"].nunique().reindex(accts, fill_value=0)
    X["n_counterparties"] = agg["counterparty"].nunique().reindex(accts, fill_value=0)
    X["incoming_endpoint_share"] = X["n_incoming"] / (X["n_incoming"] + X["n_outgoing"]).clip(lower=1)
    for name in ("cash", "cross_bank", "self_transfer"):
        X[f"{name}_share"] = agg[name].mean().reindex(accts, fill_value=0)
    X["n_currencies"] = agg["currency"].nunique().reindex(accts, fill_value=0)
    X["n_payment_formats"] = agg["payment_format"].nunique().reindex(accts, fill_value=0)
    X["active_days"] = agg["date"].nunique().reindex(accts, fill_value=0)
    span_hours = (agg["timestamp"].max() - agg["timestamp"].min()).dt.total_seconds() / 3600.0
    X["active_hours"] = span_hours.reindex(accts, fill_value=0)
    X["transactions_per_active_hour"] = X["n_transactions"] / X["active_hours"].clip(lower=1.0)
    X["mean_gap_hours"] = X["active_hours"] / (X["n_transactions"] - 1).clip(lower=1)
    hourly_max = activity.groupby(["acct", "hour_bucket"], observed=True).size().groupby(level=0, observed=True).max()
    X["max_transactions_in_clock_hour"] = hourly_max.reindex(accts, fill_value=0)
    # This is a fixed clock-hour bucket, NOT a rolling 60-minute peak detector.
    formats = sorted(activity["payment_format"].unique().tolist())
    currencies = sorted(df["currency"].unique().tolist())
    if len(formats) > 64 or len(currencies) > 64:
        raise MemoryError("More than 64 currencies or payment formats; review the categorical fields first.")
    format_map = {f"fmt_{i:02d}": value for i, value in enumerate(formats)}
    for code, value in format_map.items():
        cnt = activity.loc[activity["payment_format"].eq(value)].groupby("acct", observed=True).size()
        X[f"share_{code}"] = cnt.reindex(accts, fill_value=0) / X["n_transactions"]

    # Currency-separated amounts: never add USD to EUR as a model feature.
    out_ccy = df.groupby(["from_acct", "currency"], observed=True)["amount_paid"].agg(["sum", "mean", "max", "size"])
    in_ccy = df.groupby(["to_acct", "currency"], observed=True)["amount_paid"].agg(["sum", "mean", "max", "size"])
    out_ccy.index = out_ccy.index.set_names(["acct", "currency"])
    in_ccy.index = in_ccy.index.set_names(["acct", "currency"])
    amount_audit = out_ccy[["sum", "size"]].rename(columns={"sum": "total_out", "size": "n_out"}).join(
        in_ccy[["sum", "size"]].rename(columns={"sum": "total_in", "size": "n_in"}), how="outer").fillna(0)
    amount_audit[["n_in", "n_out"]] = amount_audit[["n_in", "n_out"]].astype("int64")
    if not np.isfinite(amount_audit.to_numpy()).all():
        raise ValueError("Per-currency amount sums overflowed. Check the input values.")
    currency_map = {f"ccy_{i:02d}": value for i, value in enumerate(currencies)}
    currency_columns = {}
    for code, value in currency_map.items():
        sums = {}
        for direction, table in (("out", out_ccy), ("in", in_ccy)):
            if value in table.index.get_level_values("currency"):
                part = table.xs(value, level="currency").reindex(accts, fill_value=0)
            else:
                part = pd.DataFrame(0.0, index=accts, columns=["sum", "mean", "max", "size"])
            currency_columns[f"{direction}_count_{code}"] = part["size"]
            for stat in ("sum", "mean", "max"):
                currency_columns[f"{direction}_log1p_{stat}_{code}"] = np.log1p(part[stat])
            sums[direction] = part["sum"]
        denominator = (sums["in"] + sums["out"]).replace(0, 1.0)
        currency_columns[f"payer_amount_balance_proxy_{code}"] = (sums["in"] - sums["out"]) / denominator

    X = pd.concat([X, pd.DataFrame(currency_columns, index=accts)], axis=1)

    # Sole label-dependent operation: make y. No y-derived field is joined to X.
    labels_out = gb_out["is_laundering"].max().reindex(accts, fill_value=0)
    labels_in = gb_in["is_laundering"].max().reindex(accts, fill_value=0)
    y = pd.Series(np.maximum(labels_out.to_numpy(), labels_in.to_numpy()), index=accts,
                  name="is_laundering", dtype="int8")
    estimated_matrix_bytes = int(X.shape[0] * X.shape[1] * 4)
    if estimated_matrix_bytes > 512 * 1024 * 1024:
        raise MemoryError("float32 feature matrix alone exceeds 512 MiB; review the slice with the team.")
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Non-finite features. Investigate; do not replace them with arbitrary values.")
    X = X.astype("float32")
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features exceeded float32 range.")
    metadata = dict(source_metadata or {})
    metadata.update({
        "transactions": int(len(df)), "accounts": int(len(accts)), "features": int(X.shape[1]),
        "laundering_transactions": int(df["is_laundering"].sum()), "positive_accounts": int(y.sum()),
        "negative_accounts": int((y == 0).sum()), "account_prevalence": float(y.mean()),
        "first_timestamp": df["timestamp"].min().isoformat(), "last_timestamp": df["timestamp"].max().isoformat(),
        "timestamp_convention": "Aware timestamps converted to UTC; naive timestamps interpreted as UTC. Confirm upstream.",
        "currencies": currencies, "currency_feature_map": currency_map, "format_feature_map": format_map,
        "mixed_currency_totals": len(currencies) > 1, "amount_warning": AMOUNT_WARNING,
        "feature_names": X.columns.tolist(), "feature_matrix_mib": estimated_matrix_bytes / (1024 ** 2),
        "evaluation_warning": EVALUATION_WARNING,
        "token_collision_note": "One-bank ownership is checked; within-bank token collisions cannot be detected from this clean file.",
    })
    return AccountData(X=X, y=y, accounts=accounts,
                       amounts_by_currency=amount_audit.reset_index()[["acct", "currency", "total_in", "total_out", "n_in", "n_out"]],
                       metadata=metadata)

### 2C · Class-weighted models and held-out predictions

In [ ]:
def resolve_backend(requested: str) -> str:
    if requested not in {"auto", "xgboost", "hist"}:
        raise ValueError("backend must be 'auto', 'xgboost', or 'hist'.")
    if requested == "hist":
        return "hist"
    try:
        from xgboost import XGBClassifier  # noqa: F401
        return "xgboost"
    except (ImportError, OSError) as exc:
        if requested == "xgboost":
            raise RuntimeError(
                "XGBoost cannot import here. Do not change the shared environment blindly. "
                "Coordinate installation or set MODEL_BACKEND='hist' for the explicit CPU fallback."
            ) from exc
        warnings.warn(
            "XGBoost unavailable: using sklearn HistGradientBoostingClassifier with training-fold "
            "sample weights. Report that backend honestly, not as XGBoost. " + str(exc), RuntimeWarning,
        )
        return "hist"


def _new_model(backend: str, config: ModelConfig, positive_weight: float, seed: int):
    if backend == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            n_estimators=config.n_estimators, max_depth=config.max_depth,
            learning_rate=config.learning_rate, min_child_weight=5,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=5.0,
            max_bin=128, scale_pos_weight=positive_weight,
            n_jobs=config.n_threads, random_state=seed, verbosity=0,
        )
    return HistGradientBoostingClassifier(
        max_iter=config.n_estimators, max_depth=config.max_depth,
        max_leaf_nodes=min(2 ** config.max_depth, 31), learning_rate=config.learning_rate,
        l2_regularization=5.0, max_bins=128, min_samples_leaf=20,
        early_stopping=False, random_state=seed,
    )


def ranking_order(accounts: Sequence[str], scores: np.ndarray) -> np.ndarray:
    # Reproducible tie break independent of the target label.
    return np.lexsort((np.asarray(accounts, dtype=str), -np.asarray(scores, dtype=float)))


def budget_row(y: np.ndarray, scores: np.ndarray, accounts: Sequence[str], k: int,
               budget_name: str) -> dict[str, Any]:
    actual_k = min(int(k), len(y))
    selected = ranking_order(accounts, scores)[:actual_k]
    hits = int(np.asarray(y)[selected].sum())
    positives = int(np.asarray(y).sum())
    prevalence = float(np.asarray(y).mean())
    precision = hits / actual_k if actual_k else 0.0
    return {"budget": budget_name, "requested_k": int(k), "selected_accounts": actual_k,
            "true_positive_accounts": hits, "precision": precision,
            "recall": hits / positives if positives else 0.0,
            "selected_fraction": actual_k / len(y),
            "lift_over_prevalence": precision / prevalence if prevalence else None}


def run_oof(data: AccountData, config: Optional[ModelConfig] = None, verbose: bool = True) -> CVResult:
    """Fit one model per fold; NEVER use a full-fit model to export OOF scores.

    Trees/hyperparameters are fixed in advance. Outer validation labels never
    pick thresholds, features, stopping rounds, or class weights. There is no
    outer-fold early stopping. Repeated manual tuning would still turn these
    results into development feedback, not an untouched final test.
    """
    config = config or ModelConfig()
    if config.n_splits < 2 or config.n_estimators < 1 or config.n_threads < 1 or config.max_depth < 1:
        raise ValueError("Need >=2 requested folds, >=1 trees, >=1 depth, and >=1 threads.")
    if config.top_k < 1 or not (0 < config.alert_fraction <= 1) or not (0 <= config.reference_threshold <= 1):
        raise ValueError("Check top_k, alert_fraction, and reference_threshold.")
    if not data.X.index.is_unique or not data.X.index.equals(data.y.index) or not data.X.index.equals(data.accounts.index):
        raise ValueError("Features, labels, and output accounts must align one-to-one.")
    if not data.y.isin([0, 1]).all() or not np.isfinite(data.X.to_numpy()).all():
        raise ValueError("Invalid labels or features.")
    y = data.y.to_numpy(dtype="int8")
    positives, negatives = int(y.sum()), int((y == 0).sum())
    minority = min(positives, negatives)
    if minority < 2:
        raise ValueError(
            f"Cannot make honest stratified OOF predictions: {positives} positive and {negatives} negative accounts. "
            "Ask Abdurub for a larger suitable clean slice containing both classes. "
            "Do NOT invent labels, relabel rows, or substitute training-set predictions."
        )
    actual_folds = min(config.n_splits, minority)
    if minority < 20:
        warnings.warn("Fewer than 20 accounts in one class. Metrics will be unstable; report actual counts.", RuntimeWarning)
    if actual_folds != config.n_splits:
        warnings.warn(f"Reducing folds from {config.n_splits} to {actual_folds} for the available minority count.")
    backend = resolve_backend(config.backend)
    if verbose:
        print(f"BACKEND: {backend}; {actual_folds} folds; {config.n_threads} CPU thread(s); no accelerator claim.")
        print(f"Accounts: {len(y):,}; positive: {positives:,}; prevalence: {y.mean():.4%}.")
        print("Evaluation: retrospective account-level OOF; NOT a past-only replay test.")
    X = data.X.to_numpy(dtype="float32", copy=False)
    ids = data.X.index.to_numpy(dtype=str)
    cv = StratifiedKFold(n_splits=actual_folds, shuffle=True, random_state=config.random_state)
    scores = np.full(len(y), np.nan, dtype="float64")
    folds = np.full(len(y), -1, dtype="int16")
    hits = np.zeros(len(y), dtype="int8")
    records, importances = [], []
    started = time.perf_counter()
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        if np.intersect1d(train_idx, valid_idx).size:
            raise AssertionError("A validation account is in its training fold.")
        train_y = y[train_idx]
        weight = float((train_y == 0).sum() / (train_y == 1).sum())
        model = _new_model(backend, config, weight, config.random_state + fold)
        fold_start = time.perf_counter()
        with threadpool_limits(limits=config.n_threads):
            if backend == "xgboost":
                model.fit(X[train_idx], train_y)
            else:
                sample_weight = np.where(train_y == 1, weight, 1.0)
                model.fit(X[train_idx], train_y, sample_weight=sample_weight)
            pred = model.predict_proba(X[valid_idx])[:, 1]
        if not np.isfinite(pred).all() or (pred < 0).any() or (pred > 1).any():
            raise ValueError("Model returned invalid risk scores.")
        scores[valid_idx], folds[valid_idx] = pred, fold
        hits[valid_idx] += 1
        binary = pred >= config.reference_threshold
        within_fold_budget = budget_row(y[valid_idx], pred, ids[valid_idx], config.top_k, "within_fold_top_k")
        row = {
            "fold": fold, "training_accounts": len(train_idx), "validation_accounts": len(valid_idx),
            "training_positive_accounts": int(train_y.sum()), "validation_positive_accounts": int(y[valid_idx].sum()),
            "training_scale_pos_weight": weight,
            "average_precision": float(average_precision_score(y[valid_idx], pred)),
            "reference_threshold": config.reference_threshold,
            "precision_at_reference_threshold": float(precision_score(y[valid_idx], binary, zero_division=0)),
            "recall_at_reference_threshold": float(recall_score(y[valid_idx], binary, zero_division=0)),
            "within_fold_top_k_actual": within_fold_budget["selected_accounts"],
            "within_fold_top_k_hits": within_fold_budget["true_positive_accounts"],
            "seconds": time.perf_counter() - fold_start,
        }
        records.append(row)
        if hasattr(model, "feature_importances_"):
            importances.append(np.asarray(model.feature_importances_, dtype=float))
        if verbose:
            print(f"  Fold {fold}/{actual_folds}: AP={row['average_precision']:.4f}; {row['seconds']:.1f}s")
        del model
        gc.collect()
    if not (hits == 1).all() or not np.isfinite(scores).all() or not (folds >= 1).all():
        raise AssertionError("Every account must receive exactly one held-out prediction.")
    audit = pd.DataFrame({"acct": ids, "risk_score": scores, "is_laundering": y, "fold": folds,
                          "held_out_prediction_count": hits})
    risk = audit[["acct", "risk_score", "is_laundering"]].copy()
    risk["total_in"] = data.accounts["total_in"].to_numpy()
    risk["total_out"] = data.accounts["total_out"].to_numpy()
    risk = risk.iloc[ranking_order(ids, scores)].reset_index(drop=True)[HANDOFF_COLUMNS]
    budget_rows = [
        budget_row(y, scores, ids, config.top_k, f"top_{config.top_k}"),
        budget_row(y, scores, ids, 10, "top_10"),
        budget_row(y, scores, ids, max(1, math.ceil(len(y) * config.alert_fraction)), "fixed_account_fraction"),
    ]
    binary = scores >= config.reference_threshold
    folds_df = pd.DataFrame(records)
    metrics = {
        "evaluation": "retrospective_account_level_oof_development",
        "backend": backend, "accounts": len(y), "positive_accounts": positives, "negative_accounts": negatives,
        "account_prevalence": float(y.mean()), "requested_folds": config.n_splits, "actual_folds": actual_folds,
        "pooled_oof_average_precision": float(average_precision_score(y, scores)),
        "mean_fold_average_precision": float(folds_df["average_precision"].mean()),
        "sd_fold_average_precision_not_a_confidence_interval": float(folds_df["average_precision"].std(ddof=1)),
        "reference_threshold": config.reference_threshold,
        "precision_at_reference_threshold": float(precision_score(y, binary, zero_division=0)),
        "recall_at_reference_threshold": float(recall_score(y, binary, zero_division=0)),
        "alerts_at_reference_threshold": int(binary.sum()),
        "top_k": budget_rows[0], "fixed_fraction_budget": budget_rows[2],
        "total_training_seconds": time.perf_counter() - started,
        "score_interpretation": "0-to-1 ranking scores from class-weighted models, not calibrated probabilities of criminality.",
        "pooled_metric_caveat": "Pooled ranking metrics mix scores from different folds; show fold AP as well. They are not identical to mean CV metrics.",
        "evaluation_warning": EVALUATION_WARNING, "amount_warning": AMOUNT_WARNING,
    }
    if importances:
        importance = pd.DataFrame({"feature": data.X.columns, "mean_fold_importance": np.mean(importances, axis=0)})
        importance = importance.sort_values("mean_fold_importance", ascending=False).reset_index(drop=True)
    else:
        importance = pd.DataFrame(columns=["feature", "mean_fold_importance"])
    return CVResult(risk, audit, folds_df, pd.DataFrame(budget_rows), importance, metrics, backend, config)

### 2D · Result files and reporting

In [ ]:
def atomic_csv(frame: pd.DataFrame, destination: str | Path) -> None:
    """Publish complete files so a teammate cannot read a half-written CSV."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".tmp", dir=destination.parent,
                                     encoding="utf-8", newline="", delete=False) as stream:
        temporary = Path(stream.name)
        try:
            frame.to_csv(stream, index=False, float_format="%.17g")
        except BaseException:
            temporary.unlink(missing_ok=True)
            raise
    try:
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)


def _json_default(obj: Any):
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Cannot JSON-encode {type(obj).__name__}")


def _atomic_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".tmp", dir=path.parent,
                                     encoding="utf-8", delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(text)
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def report_text(result: CVResult, data: AccountData) -> str:
    m = result.metrics
    b = m["top_k"]
    return f"""# Golden Hour detection report

Evaluation: **retrospective account-level out-of-fold development evaluation**.
Backend actually used: **{result.backend}**, CPU only; accelerator use not measured.

Input: {data.metadata.get('input_file', 'in-memory data')}  
Accounts: {m['accounts']:,}; label-positive accounts: {m['positive_accounts']:,}.  
Account-positive prevalence: {m['account_prevalence']:.4%}.  
Actual folds: {m['actual_folds']}.

## Results from this run

- Pooled out-of-fold average precision (AP): {m['pooled_oof_average_precision']:.6f}.
- Mean fold AP: {m['mean_fold_average_precision']:.6f}.
- Among the top {b['selected_accounts']} accounts ranked by held-out score, {b['true_positive_accounts']} are label-positive.
- Precision at that budget: {b['precision']:.2%}; recall at that budget: {b['recall']:.2%}.
- At the predeclared reference score threshold {m['reference_threshold']}: precision {m['precision_at_reference_threshold']:.2%}, recall {m['recall_at_reference_threshold']:.2%}, {m['alerts_at_reference_threshold']:,} alerts.

Average precision uses sklearn's non-interpolated definition; it is not the
trapezoidal integral under a precision-recall curve. Fold variation is not an
independent-sample confidence interval. The reference threshold is not a claim
that score 0.5 means a 50% chance of laundering.

## Permitted wording for the pitch

“On our supplied synthetic-data snapshot, {m['actual_folds']}-fold account-level
out-of-fold evaluation ranked {b['true_positive_accounts']} label-positive accounts
among the top {b['selected_accounts']}. At that review budget, precision was
{b['precision']:.1%} and recall of label-positive accounts was {b['recall']:.1%}.”

This says what was measured; it does not claim real-world detection or real money saved.

## Handoff to Abdel

Use acct and risk_score for ranking, together with the transaction graph. The
is_laundering column is evaluation ground truth only; never use it to select freezes.
The CSV preserves the exact five agreed account_risk.csv columns.

{AMOUNT_WARNING}

## Limitations that must stay with the results

{EVALUATION_WARNING}

These class-weighted risk scores are not calibrated probabilities. Labels mean
an account touched at least one labelled laundering transfer; that is not a
legal finding about a person. Account-level prevalence differs from transaction
prevalence. No independent untouched final test, scheme IDs, or real deployment
was provided. No performance result is supplied until this code actually runs.
"""


def write_results(result: CVResult, data: AccountData, output_dir: str | Path = "detection_outputs",
                  handoff_path: str | Path = "account_risk.csv") -> dict[str, str]:
    """Write audit files first and publish the requested shared handoff last."""
    output_dir, handoff_path = Path(output_dir), Path(handoff_path)
    if "input_resolved_path" in data.metadata:
        input_path = Path(data.metadata["input_resolved_path"])
        expected = (data.metadata["input_size_bytes"], data.metadata["input_mtime_ns"])
        if _file_signature(input_path) != expected:
            raise RuntimeError("Input changed during training. Freeze the clean CSV and rerun; no handoff was published.")
        if handoff_path.resolve() == input_path.resolve():
            raise ValueError("The handoff path must not overwrite the clean input file.")
    if list(result.account_risk.columns) != HANDOFF_COLUMNS:
        raise AssertionError("Incorrect downstream handoff columns.")
    if result.account_risk["acct"].duplicated().any() or not result.account_risk["risk_score"].between(0, 1).all():
        raise AssertionError("Invalid handoff accounts or scores.")
    output_dir.mkdir(parents=True, exist_ok=True)
    files = {
        "fold_metrics": output_dir / "fold_metrics.csv",
        "budget_metrics": output_dir / "budget_metrics.csv",
        "oof_audit": output_dir / "oof_audit.csv",
        "currency_audit": output_dir / "account_amounts_by_currency.csv",
        "metrics": output_dir / "metrics.json",
        "manifest": output_dir / "run_manifest.json",
        "report": output_dir / "detection_report.md",
        "handoff": handoff_path,
    }
    for frame, key in ((result.fold_metrics, "fold_metrics"), (result.budget_metrics, "budget_metrics"),
                       (result.oof_audit, "oof_audit"), (data.amounts_by_currency, "currency_audit")):
        atomic_csv(frame, files[key])
    manifest = {"created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                "configuration": asdict(result.config), "actual_backend": result.backend,
                "environment": environment_report(), "data": data.metadata,
                "model_artifact_note": "Fold models were discarded after OOF scoring to save memory. No inference service or full-fit model is claimed."}
    _atomic_text(json.dumps(result.metrics, indent=2, default=_json_default, allow_nan=False), files["metrics"])
    _atomic_text(json.dumps(manifest, indent=2, default=_json_default, allow_nan=False), files["manifest"])
    _atomic_text(report_text(result, data), files["report"])
    if not result.feature_importance.empty:
        files["feature_importance"] = output_dir / "feature_importance.csv"
        atomic_csv(result.feature_importance, files["feature_importance"])
    # A warning travels beside the CSV even when the CSV is in the shared root.
    notes = handoff_path.with_name(handoff_path.stem + "_README.txt")
    _atomic_text(AMOUNT_WARNING + "\n\n" + EVALUATION_WARNING +
                 "\n\nis_laundering is evaluation only. risk_score is not calibrated.\n", notes)
    files["handoff_notes"] = notes
    atomic_csv(result.account_risk, handoff_path)
    return {name: str(path.resolve()) for name, path in files.items()}

### 2E · Step 1 graph validation and account join

In [ ]:
# ---------------------------------------------------------------------------
# Step 1 integration: exact fields inspected in golden_hour_01_data.ipynb.
# ---------------------------------------------------------------------------
STEP1_SELECTION_WARNING = (
    "Step 1 selected ALL laundering-linked accounts plus 15,000 other accounts, "
    "then kept every transaction touching those selected accounts. Additional "
    "neighbours need not have their complete histories. It also dropped normal "
    "self-transfers while retaining laundering self-transfers. This is a "
    "label-enriched, label-dependent development snapshot, not a random or "
    "time-held-out deployment sample. Degree/activity and other features may "
    "reflect the selection procedure. OOF does NOT remove that bias. The explicit "
    "self_transfer_share feature is therefore excluded, but this does not repair "
    "the sampling design. Scores and metrics apply only to this supplied slice."
)
EVALUATION_WARNING = EVALUATION_WARNING + "\n\n" + STEP1_SELECTION_WARNING

GRAPH_COUNTS = {
    "tx_out": "n_outgoing", "tx_in": "n_incoming",
    "n_receivers": "n_sent_to", "n_senders": "n_received_from",
}
GRAPH_STRUCTURAL = ["pagerank", "has_reciprocal", "fast_forward_share", "median_hold_min"]
GRAPH_EXCLUDED_MONEY = [
    "total_in", "total_out", "log_total_in", "log_total_out", "log_out_in_gap", "out_in_ratio"
]

def available_memory_gib() -> Optional[float]:
    """Linux VM hint only; not a measured peak or a container-limit guarantee."""
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) / 1024 ** 2
    except (OSError, ValueError):
        pass
    return None


def build_step1_account_features(df: pd.DataFrame, metadata: dict[str, Any]) -> AccountData:
    data = build_account_features(df, metadata)
    # Upstream removed normal self-transfers but kept labelled self-transfers.
    # Do not explicitly learn that target-conditioned filtering rule.
    data.X = data.X.drop(columns=['self_transfer_share'])
    data.metadata.update({
        'features': int(data.X.shape[1]), 'feature_names': data.X.columns.tolist(),
        'feature_matrix_mib': data.X.size * 4 / 1024 ** 2,
        'upstream_sampling_warning': STEP1_SELECTION_WARNING,
        'explicitly_excluded_baseline_feature': 'self_transfer_share',
        'evaluation_warning': EVALUATION_WARNING,
    })
    return data


def load_step1_graph(data: AccountData, path: str | Path) -> tuple[AccountData, pd.DataFrame]:
    """Strict one-to-one account join; no labels or money totals enter X.

    Counts are also checked against this exact transaction snapshot. This catches
    common stale/sample-file errors but does not prove provenance of all features.
    The owner's source notebook was inspected: graph features there use no labels;
    its label-comparison table is printed separately and is not saved to the graph.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f'Graph file missing: {path.resolve()}. Use Abdurub\'s graph_features.csv '
            'beside the full clean_transactions.csv. No fake or silent substitute is used.'
        )
    signature = _file_signature(path)
    header = _csv_header(path)
    required = ['acct'] + list(GRAPH_COUNTS) + GRAPH_STRUCTURAL
    missing = sorted(set(required) - set(header))
    if missing:
        raise ValueError(f'Step 1 graph fields missing: {missing}. Ask Abdurub to finish his graph cells.')
    selected_logs = ['log_' + c for c in GRAPH_COUNTS if 'log_' + c in header]
    fields = required + selected_logs
    graph = pd.read_csv(path, usecols=fields, dtype={'acct': 'string'},
                        keep_default_na=False, nrows=len(data.X) + 1)
    if graph['acct'].eq('').any() or graph['acct'].str.strip().ne(graph['acct']).any():
        raise ValueError('Blank or whitespace-altered graph account codes. Do not silently rewrite them.')
    if graph['acct'].duplicated().any():
        raise ValueError('graph_features.csv must have exactly one row per acct; duplicate keys found.')
    graph = graph.set_index('acct')
    missing_accounts = data.X.index.difference(graph.index)
    extra_accounts = graph.index.difference(data.X.index)
    if len(graph) != len(data.X) or len(missing_accounts) or len(extra_accounts):
        raise ValueError(
            f'Graph and clean transaction account sets differ: {len(missing_accounts)} missing, '
            f'{len(extra_accounts)} extra (read at most account count + 1 rows). '
            'Use the full clean_transactions.csv, NOT clean_transactions_sample.csv, '
            'with the graph from the same snapshot. No accounts were dropped.'
        )
    graph = graph.reindex(data.X.index).apply(pd.to_numeric, errors='raise')
    if not np.isfinite(graph.to_numpy()).all():
        raise ValueError('Missing/infinite graph features. No zero-filling of an incomplete join.')
    for source, base in GRAPH_COUNTS.items():
        v = graph[source].to_numpy()
        if (v < 0).any() or not np.equal(v, np.floor(v)).all():
            raise ValueError(f'{source} must be a non-negative integer count.')
        if not np.array_equal(v, data.X[base].to_numpy(dtype='float64')):
            raise ValueError(f'{source} disagrees with the cleaned file: the graph is stale or from another slice.')
        log_col = 'log_' + source
        expected = np.log1p(v)
        if log_col in graph and not np.allclose(graph[log_col], expected, rtol=1e-6, atol=1e-7):
            raise ValueError(f'{log_col} does not match log1p({source}); check the Step 1 graph file.')
        if log_col not in graph:
            # An explicitly documented deterministic transform of an existing count,
            # not a guessed column or a silent missing structural feature.
            graph[log_col] = expected
    if not graph['pagerank'].between(0, 1).all():
        raise ValueError('pagerank must lie between 0 and 1.')
    if not graph['has_reciprocal'].isin([0, 1]).all():
        raise ValueError('has_reciprocal must be 0 or 1.')
    if not graph['fast_forward_share'].between(0, 1).all():
        raise ValueError('fast_forward_share must lie between 0 and 1.')
    hold = graph['median_hold_min']
    if ((hold < 0) & hold.ne(-1)).any():
        raise ValueError('median_hold_min can be non-negative or exactly -1, not another negative value.')
    if (hold.eq(-1) & graph['fast_forward_share'].gt(0)).any():
        raise ValueError('No observed forwarding (-1) conflicts with a positive fast_forward_share.')
    no_out = graph['tx_out'].eq(0)
    if (no_out & (hold.ne(-1) | graph['fast_forward_share'].ne(0))).any():
        raise ValueError('An account with no outgoing transactions cannot have observed forwarding.')
    safe = pd.DataFrame(index=data.X.index)
    for name in GRAPH_COUNTS:
        safe['log_' + name] = graph['log_' + name]
    for name in ['pagerank', 'has_reciprocal', 'fast_forward_share']:
        safe[name] = graph[name]
    # Two separate features distinguish "no forwarding observed" from a real 0-minute gap.
    safe['never_forwarded_observed'] = hold.eq(-1).astype('int8')
    safe['log1p_median_hold_min'] = np.log1p(hold.where(hold.ge(0), 0.0))
    safe = safe.add_prefix('graph__').astype('float32')
    joined = data.X.join(safe, how='left', validate='one_to_one')
    if not joined.index.equals(data.X.index) or len(joined) != len(data.X):
        raise AssertionError('The account join changed the row order or number of accounts.')
    if not np.isfinite(joined.to_numpy()).all():
        raise ValueError('Non-finite values in the final joined feature matrix.')
    fingerprint = _sha256(path)
    if _file_signature(path) != signature:
        raise RuntimeError('graph_features.csv changed while it was read; retry after the owner finishes writing.')
    metadata = dict(data.metadata)
    metadata.update({
        'features': int(joined.shape[1]), 'feature_names': joined.columns.tolist(),
        'feature_matrix_mib': joined.size * 4 / 1024 ** 2,
        'graph_file': path.name, 'graph_resolved_path': str(path.resolve()),
        'graph_sha256': fingerprint, 'graph_size_bytes': signature[0], 'graph_mtime_ns': signature[1],
        'graph_features_used': safe.columns.tolist(),
        'graph_columns_not_loaded': sorted(set(header) - set(fields)),
        'graph_monetary_columns_excluded': [c for c in GRAPH_EXCLUDED_MONEY if c in header],
        'graph_join': 'one_to_one; exact account coverage; all four counts match cleaned transactions',
        'graph_never_forwarded_accounts': int(hold.eq(-1).sum()),
        'hold_semantics': '-1 = no outgoing transaction with a prior-or-same-time receipt observed in this slice. '
                          'Step 1 matches timestamps, not individual monetary units; ties count as zero minutes.',
        'log_money_note': 'Log-transforming a mixed-currency total does not convert currencies. Only log-count '
                          'columns are used from the supplied graph; baseline monetary features stay per-currency.',
    })
    return AccountData(joined, data.y, data.accounts, data.amounts_by_currency, metadata), graph

### 2F · Large-hub review and separate ground-truth audit

In [ ]:
def audit_largest_hubs(df: pd.DataFrame, graph: pd.DataFrame, top_n: int = 10):
    """Select hubs by counts ONLY. Keep later ground-truth inspection separate.

    These files are REVIEW/AUDIT artifacts, never inputs to model fitting or freezes.
    Do not remove a hub simply because of its degree or its retrospective label.
    """
    if top_n < 1:
        raise ValueError('top_n must be positive.')
    order = graph.reset_index().sort_values(['tx_out', 'acct'], ascending=[False, True], kind='mergesort')
    hubs = order.head(top_n)['acct'].tolist()
    total_outgoing = int(graph['tx_out'].sum())
    second_largest = float(order.iloc[1]['tx_out']) if len(order) > 1 else 0.0
    review, truth, currencies = [], [], []
    for rank, acct in enumerate(hubs, 1):
        out = df.loc[df['from_acct'].eq(acct)]
        inc = df.loc[df['to_acct'].eq(acct)]
        incident = df.loc[df['from_acct'].eq(acct) | df['to_acct'].eq(acct)]
        row = graph.loc[acct]
        base = {
            'acct': acct, 'outgoing_count_rank': rank,
            'tx_out': int(row['tx_out']), 'tx_in': int(row['tx_in']),
            'n_receivers': int(row['n_receivers']), 'n_senders': int(row['n_senders']),
            'share_all_outgoing_transactions': len(out) / max(1, total_outgoing),
            'max_hub_to_runner_up_outgoing_ratio': float(row['tx_out']) / second_largest if rank == 1 and second_largest else None,
            'currency_count': int(incident['currency'].nunique()),
            'payment_formats': ', '.join(sorted(incident['payment_format'].unique().tolist())),
            'pagerank': float(row['pagerank']), 'has_reciprocal': int(row['has_reciprocal']),
            'fast_forward_share': float(row['fast_forward_share']), 'median_hold_min': float(row['median_hold_min']),
            'first_observed': incident['timestamp'].min().isoformat(),
            'last_observed': incident['timestamp'].max().isoformat(),
            'review_note': 'High-volume review candidate, NOT an automatic freeze recommendation. '
                           'Check legitimate-traffic disruption and currency-separated amounts.',
        }
        review.append(base)
        truth.append({
            'acct': acct,
            'eval_only_laundering_outgoing_tx': int(out['is_laundering'].sum()),
            'eval_only_normal_outgoing_tx': int(out['is_laundering'].eq(0).sum()),
            'eval_only_laundering_incoming_tx': int(inc['is_laundering'].sum()),
            'eval_only_normal_incoming_tx': int(inc['is_laundering'].eq(0).sum()),
            'eval_only_laundering_incident_tx': int(incident['is_laundering'].sum()),
            'eval_only_incident_tx': len(incident),
            'eval_only_laundering_outgoing_share': float(out['is_laundering'].mean()) if len(out) else 0.0,
            'warning': 'Synthetic ground truth, evaluation only; not a model feature or a selection input.',
        })
        for direction, frame in [('outgoing', out), ('incoming_payer_quoted_proxy', inc)]:
            sums = frame.groupby('currency', observed=True)['amount_paid'].agg(['size', 'sum'])
            for currency, record in sums.iterrows():
                currencies.append({'acct': acct, 'direction': direction, 'currency': currency,
                                   'tx_count': int(record['size']), 'amount_paid_sum_same_currency': float(record['sum'])})
    return pd.DataFrame(review), pd.DataFrame(truth), pd.DataFrame(currencies)

### 2G · Comparison, safe output, and charts

In [ ]:
def compare_step1_results(baseline: CVResult, connected: CVResult) -> pd.DataFrame:
    if not baseline.oof_audit[['acct', 'fold']].equals(connected.oof_audit[['acct', 'fold']]):
        raise AssertionError('Baseline and connected model were not evaluated on identical accounts/folds.')
    rows = []
    for name, result in [('account_features', baseline), ('account_plus_step1_graph', connected)]:
        m = result.metrics; b = m['top_k']; a = m['fixed_fraction_budget']
        rows.append({'model': name, 'pooled_oof_AP': m['pooled_oof_average_precision'],
                     'mean_fold_AP': m['mean_fold_average_precision'],
                     'top_k': b['selected_accounts'], 'hits_at_top_k': b['true_positive_accounts'],
                     'precision_at_top_k': b['precision'], 'recall_at_top_k': b['recall'],
                     'fixed_budget_k': a['selected_accounts'], 'fixed_budget_precision': a['precision'],
                     'fixed_budget_recall': a['recall']})
    return pd.DataFrame(rows)


def check_step1_inputs_unchanged(data: AccountData) -> None:
    for prefix in ['input', 'graph']:
        pathkey = prefix + '_resolved_path'
        if pathkey not in data.metadata:
            continue
        path = Path(data.metadata[pathkey])
        expected = (data.metadata[prefix + '_size_bytes'], data.metadata[prefix + '_mtime_ns'])
        if not path.is_file() or _file_signature(path) != expected or _sha256(path) != data.metadata[prefix + '_sha256']:
            raise RuntimeError(f'{prefix} file changed after loading. No new handoff was written; rerun on stable inputs.')


def save_connected_run(result: CVResult, data: AccountData, review: pd.DataFrame,
                       truth: pd.DataFrame, currency_audit: pd.DataFrame,
                       output_dir: str | Path, comparison: Optional[pd.DataFrame] = None) -> dict[str, str]:
    check_step1_inputs_unchanged(data)
    root = Path(output_dir)
    # Keep previous shared account_risk.csv untouched until explicitly published.
    paths = write_results(result, data, root, root / 'account_risk.csv')
    review = review.merge(result.account_risk[['acct', 'risk_score']], on='acct', how='left', validate='one_to_one')
    truth = truth.merge(result.account_risk[['acct', 'risk_score']], on='acct', how='left', validate='one_to_one')
    for name, frame in [('hub_review', review), ('hub_ground_truth_eval', truth), ('hub_amounts_by_currency', currency_audit)]:
        path = root / (name + '.csv'); atomic_csv(frame, path); paths[name] = str(path.resolve())
    if comparison is not None:
        path = root / 'graph_comparison.csv'; atomic_csv(comparison, path); paths['comparison'] = str(path.resolve())
    extra = '\n## Step 1 graph integration\n\n' + data.metadata['graph_join'] + '.\n\n'
    extra += 'Used graph features: ' + ', '.join(data.metadata['graph_features_used']) + '.\n\n'
    extra += data.metadata['log_money_note'] + '\n\n' + data.metadata['hold_semantics'] + '\n\n'
    extra += ('The connected graph model is the predeclared candidate; comparison does not silently pick a winner. '
              'No shared handoff was replaced, and no accounts were frozen. '
              'hub_ground_truth_eval.csv is retrospective ground truth and must stay out of model/selection inputs.\n')
    if len(truth):
        h = truth.iloc[0]; r = review.iloc[0]
        extra += (f'\nLargest outgoing hub: {h["acct"]}; {int(r["tx_out"]):,} outgoing transactions; '
                  f'{int(h["eval_only_laundering_outgoing_tx"]):,} labelled laundering outgoing, '
                  f'{int(h["eval_only_normal_outgoing_tx"]):,} labelled normal outgoing. '
                  'An account-positive label is not a claim that all its transactions are illicit.\n')
    _atomic_text(report_text(result, data) + extra, root / 'detection_report.md')
    return paths


def publish_connected_handoff(result: CVResult, data: AccountData, path: str | Path) -> None:
    check_step1_inputs_unchanged(data)
    path = Path(path)
    if path.resolve() in {Path(data.metadata[k]).resolve() for k in ('input_resolved_path', 'graph_resolved_path')}:
        raise ValueError('Do not overwrite an input file.')
    if list(result.account_risk.columns) != HANDOFF_COLUMNS:
        raise AssertionError('Unexpected shared handoff schema.')
    _atomic_text(AMOUNT_WARNING + '\n\n' + EVALUATION_WARNING +
                 '\n\nis_laundering is evaluation only. Scores are not calibrated.\n',
                 path.with_name(path.stem + '_README.txt'))
    atomic_csv(result.account_risk, path)


def draw_connected_charts(result: CVResult, output_dir: str | Path) -> None:
    try:
        import matplotlib.pyplot as plt
        from sklearn.metrics import precision_recall_curve
    except ImportError:
        print('matplotlib unavailable; optional charts skipped. All model/CSV results remain available.')
        return
    root = Path(output_dir); root.mkdir(parents=True, exist_ok=True)
    a = result.oof_audit
    p, r, _ = precision_recall_curve(a['is_laundering'], a['risk_score'])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.step(r, p, where='post', label='Account + Step 1 graph, OOF')
    ax.axhline(a['is_laundering'].mean(), linestyle='--', label='Selected-slice account prevalence')
    ax.set(xlabel='Recall of label-positive accounts', ylabel='Precision', xlim=(0, 1), ylim=(0, 1.02),
           title='Retrospective OOF: label-enriched Step 1 slice')
    ax.legend(); fig.tight_layout(); fig.savefig(root / 'oof_precision_recall.png', dpi=160)
    plt.show(); plt.close(fig)
    ordered = result.account_risk
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(np.arange(1, len(ordered) + 1) / len(ordered),
            ordered['is_laundering'].cumsum() / ordered['is_laundering'].sum())
    ax.set(xlabel='Fraction of accounts reviewed in score order', ylabel='Recall of label-positive accounts',
           title='Offline review budget; not a time-to-freeze curve', xlim=(0, 1), ylim=(0, 1.02))
    fig.tight_layout(); fig.savefig(root / 'oof_review_budget.png', dpi=160)
    plt.show(); plt.close(fig)

## 3 · Load the full cleaned file
Both input files must exist. A sample/full-graph mismatch is rejected rather than
silently dropping accounts. Identifiers remain pseudonymised and are never used as
model features. Naive timestamps follow the original project contract’s UTC
interpretation; confirm that convention with the data owner.

In [ ]:
from IPython.display import display, Markdown
print(json.dumps(environment_report(), indent=2))
missing = [str(path) for path in (INPUT_PATH, GRAPH_PATH) if not path.is_file()]
if missing:
    raise FileNotFoundError("Required Step 1 file(s) missing: " + ", ".join(missing))
available = available_memory_gib()
if available is not None:
    print(f"OS-reported available memory: {available:.2f} GiB (hint, not a peak-use guarantee)")
    if available < MIN_AVAILABLE_RAM_GIB:
        raise MemoryError("Shared memory is low. Coordinate with teammates before running the full pipeline.")
transactions, source_metadata = load_transactions(INPUT_PATH, MAX_TRANSACTIONS)
print(f"Loaded {len(transactions):,} transactions from the full cleaned CSV.")
print("Observed currencies:", sorted(transactions["currency"].unique().tolist()))
print("Time range:", transactions["timestamp"].min(), "to", transactions["timestamp"].max())
print("Input SHA-256:", source_metadata["input_sha256"])
display(transactions["is_laundering"].value_counts().rename_axis("label").to_frame("transactions"))

## 4 · Build account features and join Step 1 on `acct`
The join is one-to-one, preserves every account, and checks all four graph counts
against the cleaned transactions. It uses the graph’s **log-count columns**, PageRank,
reciprocal-link flag and forwarding-time features. Unknown columns and labels are
not automatically added.

**Mixed currencies:** raw `total_in`/`total_out` are not model features. Nor are the
graph’s `log_total_in`, `log_total_out` or `log_out_in_gap`: taking logs is not currency
conversion. The baseline’s amount features stay separated by currency. Raw totals
are retained **only** in the agreed output contract, with a warning and currency audit.

**`median_hold_min = -1`:** encoded as `never_forwarded_observed = 1`, plus a zero-filled
log-duration. A genuine zero-minute gap has that indicator set to zero. This describes
observed ordering in this slice, not proof that particular funds were forwarded.

**Upstream filtering:** Step 1 removed normal self-transfers but kept laundering
self-transfers. The explicit `self_transfer_share` predictor is excluded to avoid
learning that filtering rule. Other selection bias remains; the report explains it.

In [ ]:
baseline_data = build_step1_account_features(transactions, source_metadata)
connected_data, graph_table = load_step1_graph(baseline_data, GRAPH_PATH)
print(f"Accounts: {len(connected_data.X):,}; label-positive accounts: {int(connected_data.y.sum()):,}")
print(f"Features: {baseline_data.X.shape[1]} baseline + {connected_data.X.shape[1] - baseline_data.X.shape[1]} graph")
print("Verified join:", connected_data.metadata["graph_join"])
print("Graph features used:", connected_data.metadata["graph_features_used"])
print("Excluded mixed-currency graph fields:", connected_data.metadata["graph_monetary_columns_excluded"])
print("Accounts with median_hold_min = -1:", connected_data.metadata["graph_never_forwarded_accounts"])
print("Graph SHA-256:", connected_data.metadata["graph_sha256"])
print(STEP1_SELECTION_WARNING)

## 5 · Inspect the large hubs — no freeze decisions
The largest hubs are selected for review using transaction counts **only**. Model
training does not read the audit’s ground-truth columns. The later evaluation file
shows how much labelled-normal traffic would be affected; it is not an input for
ranking or selecting accounts. No hub is automatically removed or frozen.

In [ ]:
hub_review, hub_truth, hub_currencies = audit_largest_hubs(transactions, graph_table, top_n=10)
display(hub_review[["acct", "tx_out", "tx_in", "n_receivers", "n_senders",
                    "currency_count", "share_all_outgoing_transactions"]])
print("Largest hub kept for review, not automatically frozen:", hub_review.iloc[0]["acct"])
# Free the transaction table before training. Neither input file is changed.
del transactions
gc.collect()

## 6 · Train with three held-out folds
Each account receives one score from a model that excluded that account row. Features
summarise the whole snapshot, related accounts may cross folds, and the data selection
used labels. Do not describe this as leakage-free end-to-end validation, a future
prediction test, or a per-bank private model.

The baseline and graph version use identical accounts and fold assignments. The graph
version is the predeclared review candidate; the code does not silently switch models
based on the results. A lower comparison score must be reported honestly.

In [ ]:
config = ModelConfig(backend=MODEL_BACKEND, n_splits=N_SPLITS, n_estimators=N_ESTIMATORS,
                     n_threads=N_THREADS, random_state=RANDOM_SEED, top_k=TOP_K,
                     alert_fraction=ALERT_FRACTION)
baseline_result = None
comparison = None
if RUN_BASELINE_COMPARISON:
    print("BASELINE — account features only")
    baseline_result = run_oof(baseline_data, config)
print("CONNECTED — account features plus Step 1 graph")
result = run_oof(connected_data, config)
assert result.oof_audit["held_out_prediction_count"].eq(1).all()
if baseline_result is not None:
    comparison = compare_step1_results(baseline_result, result)
    display(comparison)
print("Verified: every account has exactly one held-out prediction.")

## 7 · Save a review copy and inspect the results
This writes inside `detection_outputs_step1/`, not over the team’s shared handoff.
Metrics include average precision (AP), precision/recall at the top-100 budget and
at a fixed 0.5% account-review budget, plus per-fold results. A class-weighted score
is not a calibrated probability or a finding of guilt.

The **normal-versus-laundering hub breakdown below is retrospective ground truth**.
It is deliberately separated from the model and selection inputs.

In [ ]:
paths = save_connected_run(result, connected_data, hub_review, hub_truth, hub_currencies,
                           OUTPUT_DIR, comparison)
if baseline_result is not None:
    write_results(baseline_result, baseline_data, OUTPUT_DIR / "baseline",
                  OUTPUT_DIR / "baseline" / "account_risk.csv")
display(result.fold_metrics)
display(result.budget_metrics)
display(result.account_risk.head(10))
display(hub_truth.head(3))
print("Review copy saved:", paths["handoff"])
print("Model comparison:", paths.get("comparison", "comparison disabled"))
print("Hub review:", paths["hub_review"])
print("Ground-truth-only hub audit:", paths["hub_ground_truth_eval"])
print("Currency-separated audit:", paths["currency_audit"])
print("Report:", paths["report"])
print("Shared account_risk.csv has NOT been replaced.")
display(Markdown(Path(paths["report"]).read_text()))

## 8 · Optional result charts
Each chart uses the actual held-out predictions from this run. These are offline
review-budget charts, not a claim of money stopped at earlier times.

In [ ]:
if MAKE_PLOTS:
    draw_connected_charts(result, OUTPUT_DIR)
else:
    print("Optional charts skipped.")

## 9 · Publish only when you decide
**In this copy, publishing is already enabled:** `PUBLISH_TO_SHARED = True` in Settings.
Running all notebook cells, including the last publishing cell, writes or replaces
`/home/ibm-user/shared/account_risk.csv` after training and validation succeed.
Do not change the input paths between training and publishing. Publishing checks
both input file fingerprints again. If a shared file already exists, it is replaced.

The five agreed columns remain `acct, risk_score, is_laundering, total_in, total_out`.
The last two contain unconverted original-unit sums for compatibility, **not**
comparable money values. `is_laundering` is evaluation-only.

In [ ]:
if PUBLISH_TO_SHARED:
    publish_connected_handoff(result, connected_data, SHARED_DIR / "account_risk.csv")
    print("Published:", SHARED_DIR / "account_risk.csv")
else:
    print("Not published. Your review copy is:", OUTPUT_DIR / "account_risk.csv")
    print("No shared handoff replaced and no accounts frozen.")

In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import platform

print("Architecture:", platform.machine())

cpuinfo_path = Path("/proc/cpuinfo")
if cpuinfo_path.exists():
    cpuinfo = cpuinfo_path.read_text(errors="replace").lower()
    feature_lines = [
        line.split(":", 1)[1]
        for line in cpuinfo.splitlines()
        if ":" in line
        and line.split(":", 1)[0].strip() in {"features", "flags"}
    ]
    flags = set(" ".join(feature_lines).split())
    print("NNPA flag advertised:", "nnpa" in flags if feature_lines else "Unknown")
else:
    print("NNPA flag advertised: Unknown")

for package in ("tensorflow", "ibm-zdnn-plugin", "xgboost"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not found in this notebook's Python environment")

print("This checks availability only—not whether a model uses the accelerator.")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Avoid starting TensorFlow when the shared server is short of memory.
meminfo = Path("/proc/meminfo")
if meminfo.exists():
    for line in meminfo.read_text().splitlines():
        if line.startswith("MemAvailable:"):
            available_gib = int(line.split()[1]) / (1024 ** 2)
            print(f"Available system memory: {available_gib:.2f} GiB")
            if available_gib < 1.5:
                raise RuntimeError(
                    "Not enough spare memory for this optional check. "
                    "Wait until teammates finish their heavy jobs."
                )

# These settings affect only the temporary test process.
probe_env = os.environ.copy()
probe_env.update({
    "NNPA_DEVICES": "1",
    "ZDNN_ENABLE_PRECHECK": "true",
    "TF_CPP_MIN_LOG_LEVEL": "0",
    "TF_CPP_VMODULE": "remapper=1",
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "TF_NUM_INTRAOP_THREADS": "1",
    "TF_NUM_INTEROP_THREADS": "1",
})

probe_code = r'''
import tensorflow as tf

print("TensorFlow imported:", tf.__version__, flush=True)
print("Visible devices:", tf.config.list_physical_devices(), flush=True)

# Graph execution is needed for IBM's operation-remapping optimisations.
@tf.function(autograph=False)
def test_calculation(a, b):
    return tf.nn.relu(tf.matmul(a, b))

a = tf.ones((256, 256), dtype=tf.float32)
b = tf.ones((256, 256), dtype=tf.float32)
result = test_calculation(a, b).numpy()

print("Calculation correct:", bool((result == 256.0).all()), flush=True)
print("Check the diagnostic logs for accelerator use.", flush=True)
'''

print("Starting the diagnostic—no training or CSV changes.", flush=True)

try:
    probe = subprocess.run(
        [sys.executable, "-u", "-c", probe_code],
        env=probe_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=120,
    )
    print(probe.stdout)
    print("Process exit code:", probe.returncode)
except subprocess.TimeoutExpired:
    print("The diagnostic timed out and was stopped. Do not rerun everything.")

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import xgboost

path = Path("/home/ibm-user/shared/account_risk.csv")
expected = "b9060d54b71da0fb8592a29c5c39ca9788f77dabfffd21051f3a0a9c04a7e0ad"

if not path.exists():
    print("ERROR: account_risk.csv not found")
else:
    actual = hashlib.sha256(path.read_bytes()).hexdigest()

    print("Actual SHA-256:", actual)
    print("HASH MATCH:", actual == expected)

    df = pd.read_csv(path)
    print("Account rows:", len(df))
    print("Unique accounts:", df["acct"].nunique())
    print("Scores valid:", df["risk_score"].between(0, 1).all())
    print("XGBoost version:", xgboost.__version__)

## Sources and interpretation
Project basis: uploaded `Golden_Hour_Team_Plan.pdf` (Alex’s account-level detection
assignment) and `golden_hour_01_data.ipynb` (actual Step 1 schema, graph construction,
sampling and sentinel definitions). The original 18-page specification’s per-bank,
past-only transaction scoring is a different workflow and is not claimed here.

Graph source definitions are preserved: PageRank uses the unweighted directed graph
of distinct non-self links; `has_reciprocal` means a two-way link, not detection of
all laundering cycles; holding time uses the most recent receipt at or before each
outgoing timestamp and does not trace individual units of money.

The explicit graph feature allowlist, encoding of missing forwarding, exclusion of
mixed-currency log sums, self-transfer predictor exclusion, memory settings, review
folder and audit files are implementation choices in this notebook. Step 1 is not
modified. No official AML rule, live bank feed, real identity, legal freeze or Telum
acceleration is claimed.

Official API references checked during integration:
- [pandas one-to-one joins](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)
- [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html)
- [scikit-learn average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
- [scikit-learn out-of-fold prediction caveat](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)

The graph CSV itself was not uploaded to this chat during preparation. Local testing
reconstructed it by running only the two graph cells from the uploaded Step 1 notebook
on the full uploaded clean CSV. That does not verify the separate copy on your server.
This notebook instead reads and validates the existing CSV in your shared folder.